# 4.5 FA 算子运行与调测：CANNSim 仿真 + DebugTools

本节是第 4 章的高潮：把 FA 算子在仿真环境里完整跑一遍，走完 **tiling 编译 → tiling 运行 → kernel 编译 → kernel 运行 → dump 对拍** 的标准调测流程。

![调测全流程](images/debug_flow.svg)

## 1. 调测的核心方法论

调 FA 算子的第一原则，来自 **mskpp 工具** 的设计思想：

> **tiling 与 kernel 分开，编译与运行分开。**

为什么？FA 是三层结构（调用通路 / Tiling / Kernel）的融合体，任何一层都可能出错：

- tiling 算错了基本块 → kernel 拿到错误的表格，跑得再对也没用；
- kernel 四拍循环逻辑错了 → tiling 给的表再正确也白搭；
- 输入构造错了（golden 本身不对）→ 前两者都白调。

分层隔离后，问题域大幅缩小：**tiling 单独调通（看 tiling 日志），kernel 单独调通（看 dump），最后合到一起跑全流程**——定位故障时不用大海捞针。

## 2. 准备调测输入：golden 与 json

调测需要两样输入：

| 文件 | 内容 | 说明 |
|--|--|--|
| **golden（bin 格式）** | 标准答案数据 | 用 numpy / torch 按 FA 数 学公式生成，序列化为二进制 |
| **json** | 算子属性信息 | tensor 的 shape、dtype、精度对比标准（如相对误差阈值）等 |

```python
# 生成 golden 的思路（示例）
import numpy as np

def golden_attention(q, k, v, scale):      # q:[N1,D] k,v:[N2,D] → 按 GQA 广播
    scores = q @ k.T * scale              # S = Q·Kᵀ · scale
    scores = scores - scores.max(-1, keepdims=True)
    p = np.exp(scores); p /= p.sum(-1, keepdims=True)   # softmax
    return p @ v                          # P·V

q = np.random.randn(32, 128).astype(np.float16)
# ... 构造 k/v（GQA 广播到 32 头）后调用，结果 tofile("golden.bin")
```

> golden 用 **fp32 中间过程** 生成、最后转 fp16，与算子的 softmaxFlash 内部精度对齐，避免对拍时的无谓误差。

## 3. 四步调测流程

### 步骤①②：Tiling 先编译、再运行

```bash
# DebugTools 工程目录下（tests/debug_tools/）
bash run.sh -t compile      # tiling 编译（CPU 侧，产物是可执行/库）
bash run.sh -t run         # tiling 运行：真实执行 DoTiling 八步

# 查看产物：tiling 运行结果在
ls ascend_tools/debug_workspace/<算子名>/tiling/
```

在这里验证第 3 章 3.5 节学的一切：基本块选了多大、`totalSize` / `splitFactorSize` 是多少、TilingKey 是什么值。**tiling 日志是阅读真实源码最好的「观测窗口」。**

### 步骤③：Kernel 编译成两个 .o

```bash
bash run.sh -k compile

# 产物（分离架构）：kernel 有 AIC 和 AIV 的区别，
# 所以会编译成两个 .o：
#   <算子名>_aic.o   Cube 核代码
#   <算子名>_aiv.o   Vector 核代码
```

### 步骤④：Kernel 运行（仿真）

```bash
bash run.sh -k run

# 运行过程中，用 tail 命令查看实时日志：
tail -f ascend_tools/debug_workspace/<算子名>/simulator/*.log
```

在 kernel 代码里可以用 **DumpTensor** 输出中间结果观察：

```cpp
// 第二个参数是一个标识 index，用来区分同一段缓冲的不同次 dump
DumpTensor(softmaxResult, 0);   // dump 第一次（分块前的状态）
DumpTensor(softmaxResult, 1);   // dump 第二次（更新后的状态）
```

## 4. 取回结果并对拍

```bash
# 运行完成后，在 dump 目录下找 dump 值（运行结果）：
ls ascend_tools/debug_workspace/<算子名>/simulator/dump/

# 用对拍脚本按 json 里的精度标准比对 dump vs golden
python compare.py dump/golden.bin dump/atten_mask.bin --rtol 2e-3
```

| 结果 | 判断 | 下一步 |
|--|--|--|
| 一致 | ✅ 正确性通过 | 进入第 5 章：亲手写一个 |
| 不一致 | ❌ 分层定位 | 先查 tiling 日志参数是否合理 → 再用 DumpTensor 逐拍 dump，定位到 mm1/vector1/mm2/vector2 哪一拍开始错 |

> 逐拍 dump 的位置就是第 3 章 3.7 节四拍循环的观测点——调测工具与代码结构是一一对应的。

## 5. 调测 checklist（小结）

```text
□ 环境就绪（4.2 节三路径之一）
□ golden 用 fp32 中间过程生成，bin 格式
□ json 写清 shape / dtype / 精度标准
□ tiling 编译 + 运行，检查 debug_workspace/<op>/tiling/ 的参数
□ kernel 编译出 aic / aiv 两个 .o
□ kernel 运行，tail -f 盯实时日志
□ dump 目录取回结果，按精度标准对拍
□ 失败时：先 tiling 后 kernel，DumpTensor(index) 逐拍定位
```

## 小测验

1. mskpp 工具「四个分开」指什么？分别解决什么定位难题？
2. kernel 编译为什么产出两个 `.o`？运行时它们如何配合？
3. dump 对拍失败时，你的排查顺序是什么？为什么 tiling 排在 kernel 前面？